In [100]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenizer 및 임베딩 모델 로드 (LLM2Vec)
tokenizer_embed = AutoTokenizer.from_pretrained("McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp")
# padding token이 없어서 eos_token을 padding token으로 설정
tokenizer_embed.pad_token = tokenizer_embed.eos_token
# Quantization 설정 (4-bit)
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
# LoRA를 사용한 파라미터 효율적 파인튜닝 설정 (PeftModel)
lora_config = LoraConfig(
    r=8,  # 랭크
    lora_alpha=16,  # alpha 값 (LoRA 계층에서의 스케일링)
    lora_dropout=0.05,  # 드롭아웃 비율
    bias="none",  # bias는 학습하지 않음
    task_type="SEQ2SEQ_LM"  # 비지도 학습의 경우 task_type을 SEQ2SEQ로 변경
)
# Quantized 인코더 모델 로드
model_embed = AutoModel.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

model = get_peft_model(model_embed, lora_config)
model.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]


PeftModel(
  (base_model): LoraModel(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 4096)
      (layers): ModuleList(
        (0-31): 32 x LlamaDecoderLayer(
          (self_attn): LlamaSdpaAttention(
            (q_proj): lora.Linear4bit(
              (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.05, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=4096, out_features=8, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=8, out_features=4096, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (k_proj): lora.Linear4bit(
              (base_layer): Linear4bit(in_features=4096, out_feature

In [2]:
# SQuAD 같은 텍스트 데이터셋을 사용한 예시 (TriviaQA도 가능)
dataset = load_dataset("squad", split="train[:1000]")  # 예를 들어 1000개의 샘플만 사용

In [143]:
# 80%는 훈련용, 20%는 검증용으로 분리
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

In [144]:
# 배치 처리에서 질문과 문서를 한꺼번에 묶지 않고 따로 임베딩을 처리한 후, 이를 비교하는 방식으로 변경합니다.
class CustomTextDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        question = self.dataset[idx]['question']
        context = self.dataset[idx]['context']

        # 질문 인코딩
        question_input = self.tokenizer.encode_plus(
            question,
            add_special_tokens=True,
            return_tensors='pt',
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # 문서 인코딩
        context_input = self.tokenizer.encode_plus(
            context,
            add_special_tokens=True,
            return_tensors='pt',
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # 각 텐서를 반환 (차원에 맞게 반환)
        return {
            'question_input_ids': question_input['input_ids'].squeeze(),
            'question_attention_mask': question_input['attention_mask'].squeeze(),
            'context_input_ids': context_input['input_ids'].squeeze(),
            'context_attention_mask': context_input['attention_mask'].squeeze()
        }



In [145]:
train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 800
})

In [146]:
# 훈련용 및 검증용 데이터셋 생성
train_dataset = CustomTextDataset(train_dataset, tokenizer_embed)
eval_dataset = CustomTextDataset(eval_dataset, tokenizer_embed)

In [147]:
# TrainingArguments 설정 (비지도 학습이므로 loss 계산 없이 진행)
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_dir='./logs',  # 로그 저장 경로
    logging_steps=10,  # 매 10 스텝마다 로깅
    save_steps=100,  # 모델을 저장할 스텝 주기
    evaluation_strategy="steps",  # 주기적인 평가를 위한 설정
    eval_steps=50,  # 매 50 스텝마다 평가
    save_total_limit=2,  # 체크포인트 저장 개수 제한
    load_best_model_at_end=True,  # 최종적으로 가장 좋은 모델 로드
    logging_first_step=True,  # 첫 스텝부터 로깅
    logging_strategy="steps",  # 스텝 단위로 로깅
    report_to="none",  # 리포팅 비활성화 (기본적으로 콘솔에 출력)
    fp16=True  # 혼합 정밀도 사용
)


/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [148]:
from torch.nn import functional as F

# Cosine Similarity Loss 계산 함수
def compute_cosine_loss(model, inputs):
    # 질문과 문서의 input_ids 및 attention_mask를 따로 처리
    question_input_ids = inputs["question_input_ids"].unsqueeze(0)  # 배치 차원 추가
    question_attention_mask = inputs["question_attention_mask"].unsqueeze(0)
    context_input_ids = inputs["context_input_ids"].unsqueeze(0)  # 배치 차원 추가
    context_attention_mask = inputs["context_attention_mask"].unsqueeze(0)

    # 질문 임베딩 추출
    question_outputs = model(input_ids=question_input_ids, attention_mask=question_attention_mask)
    question_embeddings = question_outputs.last_hidden_state.mean(dim=1)  # 문장 임베딩

    # 문서 임베딩 추출
    context_outputs = model(input_ids=context_input_ids, attention_mask=context_attention_mask)
    context_embeddings = context_outputs.last_hidden_state.mean(dim=1)  # 문장 임베딩

    # 질문과 문서 간의 코사인 유사도 계산
    similarities = F.cosine_similarity(question_embeddings, context_embeddings, dim=-1)
    
    # 유사도를 최대화하는 방향으로 손실 계산
    loss = 1 - similarities.mean()  # 유사도가 높을수록 손실이 줄어듬
    return loss



# Trainer 설정 및 loss 계산을 위한 커스터마이징
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Cosine Similarity 기반의 손실 함수 호출
        loss = compute_cosine_loss(model, inputs)
        return (loss, None) if return_outputs else loss

In [149]:
from transformers import TrainerCallback
# Custom Logging Callback 정의
class CustomLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step: {state.global_step}")
            if 'loss' in logs:
                print(f"Training Loss: {logs['loss']}")
            if 'eval_loss' in logs:
                print(f"Validation Loss: {logs['eval_loss']}")
            print('-' * 50)

In [150]:
# Trainer에 콜백 추가
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # 검증용 데이터셋이 필요
    callbacks=[CustomLoggingCallback()]  # 콜백 추가
)

# 모델 학습 실행
trainer.train()


/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


ValueError: The batch received was empty, your model won't be able to train on it. Double-check that your training dataset contains keys expected by the model: input_ids,attention_mask,position_ids,past_key_values,inputs_embeds,use_cache,output_attentions,output_hidden_states,return_dict,cache_position,label,label_ids.

In [26]:
def get_embeddings(texts, model, tokenizer, device):
    # 입력 텍스트를 토크나이저로 처리
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # 모델 평가 모드로 전환 (추론 시)
    model.eval()
    
    with torch.no_grad():
        # 모델의 출력에서 hidden_states를 얻기 위해 output_hidden_states=True 설정
        outputs = model(**inputs, output_hidden_states=True)
        
        # hidden_states는 모든 레이어의 히든 상태를 포함하며, 마지막 레이어를 사용
        hidden_states = outputs.hidden_states[-1]  # 마지막 레이어의 hidden_states
        
        # 문장 수준 임베딩을 얻기 위해 마지막 hidden state의 평균을 계산
        embeddings = hidden_states.mean(dim=1)  # batch_size x hidden_dim 크기
    return embeddings

# 임베딩 생성 예시
texts = ["임베딩을 생성할 문장을 여기에 입력합니다."]
embeddings = get_embeddings(texts, model, tokenizer_embed, device)

# 생성된 임베딩 출력
print(embeddings)


tensor([[ 0.1909, -1.0954, -0.4794,  ..., -0.9416,  1.2193, -0.1249]],
       device='cuda:1')
